# 03 — Thermal Drone Detector

Fine-tune a fixed YOLO detector on the **corrected thermal-drone annotations**. The detector is an external evaluation instrument for the super-resolution study; it is not part of the proposed SR method.

**Protocol:** train on `train`, use `valid` for model selection, save the selected `best.pt` permanently in Google Drive, then freeze the detector. The closed `test` split remains untouched until the SR method and evaluation protocol are frozen.


## 1. Persistent configuration

All expensive training artifacts are written to Google Drive so a Colab disconnect does not erase the experiment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess, shutil, json

PROJECT_REPO = 'https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DATASET_ROOT = Path('/content/drive/MyDrive/thermal-drone-dataset')
DETECTOR_ROOT = DATASET_ROOT / 'detector'
RUNS_DIR = DETECTOR_ROOT / 'runs'
WEIGHTS_DIR = DETECTOR_ROOT / 'weights'
for p in [DETECTOR_ROOT, RUNS_DIR, WEIGHTS_DIR]: p.mkdir(parents=True, exist_ok=True)

SEED = 12345
MODEL = 'yolo11n.pt'  # compact pretrained detector; keep fixed once the baseline is frozen
IMGSZ = 640
EPOCHS = 100
BATCH = 16
RUN_NAME = 'thermal_drone_yolo11n_corrected_v1'
print('Persistent detector root:', DETECTOR_ROOT)


## 2. Fetch corrected annotations and install Ultralytics


In [ ]:
project_dir = Path('/content/target-aware-infrared-sr')
if not project_dir.exists():
    subprocess.run(['git','clone',PROJECT_REPO,str(project_dir)],check=True)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=project_dir,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'],check=True)
from ultralytics import YOLO
import ultralytics
print('Ultralytics:', ultralytics.__version__)


## 3. Prepare dataset images and corrected labels

The Drive layout is asymmetric: `train` and `test` are stored as ZIP archives, while `valid` is already extracted. ZIP archives are extracted only to temporary Colab storage (`/content`); persistent detector weights/results remain in Drive.


In [ ]:
import zipfile

# Keep image working copies in fast, temporary Colab storage.
LOCAL_IMAGES = Path('/content/thermal_drone_detector_images')
LOCAL_IMAGES.mkdir(parents=True, exist_ok=True)
YOLO_DATA = Path('/content/thermal_drone_yolo_dataset')
YOLO_DATA.mkdir(parents=True, exist_ok=True)

splits = ['train', 'valid', 'test']
exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def prepare_images(split):
    src_dir = DATASET_ROOT / split
    direct_images = [p for p in src_dir.iterdir()
                     if p.is_file() and p.suffix.lower() in exts]
    if direct_images:
        # valid is stored extracted on Drive, but training/validation should read
        # from fast local Colab storage. Copy it once per runtime.
        local_dir = LOCAL_IMAGES / split
        marker = local_dir / '.complete'
        if not marker.exists():
            if local_dir.exists():
                shutil.rmtree(local_dir)
            local_dir.mkdir(parents=True, exist_ok=True)
            print(f'{split}: copying {len(direct_images)} images from Drive to temporary Colab storage...')
            for src in direct_images:
                shutil.copy2(src, local_dir / src.name)
            marker.touch()
        local_images = [p for p in local_dir.iterdir()
                        if p.is_file() and p.suffix.lower() in exts]
        print(f'{split}: using {len(local_images)} local images from {local_dir}')
        return local_dir

    zip_path = src_dir / f'images_{split}.zip'
    if not zip_path.exists():
        zips = list(src_dir.glob('*.zip'))
        if len(zips) == 1:
            zip_path = zips[0]
        else:
            raise FileNotFoundError(f'No images or unique ZIP found for {split}: {src_dir}')

    extract_root = LOCAL_IMAGES / split
    marker = extract_root / '.complete'
    if not marker.exists():
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f'{split}: extracting {zip_path.name} to temporary Colab storage...')
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_root)
        marker.touch()

    candidates = [extract_root]
    candidates += [p for p in extract_root.rglob('*') if p.is_dir()]
    for p in candidates:
        imgs = [x for x in p.iterdir() if x.is_file() and x.suffix.lower() in exts]
        if imgs:
            print(f'{split}: found {len(imgs)} images in {p}')
            return p
    raise FileNotFoundError(f'ZIP extracted but no images found for {split}')

for split in splits:
    src_images = prepare_images(split)
    src_labels = project_dir / 'annotations' / 'corrected_yolo' / split / 'labels'
    assert src_labels.exists(), src_labels

    dst_images = YOLO_DATA / split / 'images'
    dst_labels = YOLO_DATA / split / 'labels'
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    # Local symlinks avoid duplicating thousands of images.
    for src in src_images.iterdir():
        if src.is_file() and src.suffix.lower() in exts:
            dst = dst_images / src.name
            if not dst.exists():
                dst.symlink_to(src)

    # Corrected labels are copied into the local YOLO workspace.
    for src in src_labels.glob('*.txt'):
        dst = dst_labels / src.name
        if not dst.exists() or dst.read_bytes() != src.read_bytes():
            shutil.copy2(src, dst)

    print(split,
          'images=', len([p for p in dst_images.iterdir() if p.suffix.lower() in exts]),
          'labels=', len(list(dst_labels.glob('*.txt'))))


## 4. Create `dataset.yaml` and validate split counts

All drone annotation classes are mapped to a single detection class (`drone`) for the downstream target-preservation evaluation. The original corrected annotation geometry is retained.


In [ ]:
# Convert class IDs to a single drone class in the experiment copy only.
for split in splits:
    for lp in (YOLO_DATA/split/'labels').glob('*.txt'):
        lines=[]
        for line in lp.read_text().splitlines():
            p=line.split()
            if len(p)>=5: lines.append('0 '+' '.join(p[1:5]))
        lp.write_text(('\n'.join(lines)+'\n') if lines else '')

yaml_path=YOLO_DATA/'dataset.yaml'
yaml_path.write_text(f'''path: {YOLO_DATA}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n  0: drone\n''')

expected={'train':2809,'valid':2115,'test':2108}
for split,n in expected.items():
    ni=len([p for p in (YOLO_DATA/split/'images').iterdir() if p.suffix.lower() in exts])
    nl=len(list((YOLO_DATA/split/'labels').glob('*.txt')))
    assert ni==n, f'{split}: expected {n} images, found {ni}'
    assert nl==n, f'{split}: expected {n} label files, found {nl}'
print(yaml_path.read_text())
print('✓ Dataset counts validated; test remains closed for training/model selection.')


## 5. Train / resume on corrected `train`, select on `valid`

If a previous `last.pt` exists in the persistent run directory, training resumes from it. Otherwise training starts from the fixed pretrained checkpoint.


In [ ]:
run_dir=RUNS_DIR/RUN_NAME
last_pt=run_dir/'weights'/'last.pt'
if last_pt.exists():
    print('Resuming:',last_pt)
    model=YOLO(str(last_pt))
    train_results=model.train(resume=True)
else:
    model=YOLO(MODEL)
    train_results=model.train(data=str(yaml_path),epochs=EPOCHS,imgsz=IMGSZ,batch=BATCH,device=0,seed=SEED,deterministic=True,project=str(RUNS_DIR),name=RUN_NAME,exist_ok=True,patience=20,workers=2,plots=True)

best_pt=run_dir/'weights'/'best.pt'
assert best_pt.exists(), f'best.pt not found: {best_pt}'
frozen_best=WEIGHTS_DIR/f'{RUN_NAME}_best.pt'
shutil.copy2(best_pt,frozen_best)
print('✓ Frozen detector:',frozen_best)


## 6. Validation metrics for the frozen detector

These are detector-development metrics on the validation split. They are **not** final test results.


In [ ]:
import pandas as pd
frozen_best=WEIGHTS_DIR/f'{RUN_NAME}_best.pt'
assert frozen_best.exists(), frozen_best
model=YOLO(str(frozen_best))
val=model.val(data=str(yaml_path),split='val',imgsz=IMGSZ,device=0,plots=True,project=str(RUNS_DIR),name=RUN_NAME+'_frozen_valid',exist_ok=True)
summary={'model':RUN_NAME,'split':'valid','n_images':2115,'precision':float(val.box.mp),'recall':float(val.box.mr),'map50':float(val.box.map50),'map50_95':float(val.box.map)}
summary_path=DETECTOR_ROOT/'validation_summary.json'
summary_path.write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Saved:',summary_path)


## 7. Freeze experiment record

After this point, do not retrain or choose another detector based on SR comparison results. The same frozen checkpoint must be used for Original DifIISR and all target-aware variants.


In [ ]:
record={'purpose':'Fixed downstream detector for SR evaluation','model':RUN_NAME,'base_checkpoint':MODEL,'seed':SEED,'imgsz':IMGSZ,'epochs_requested':EPOCHS,'train_split':'train','selection_split':'valid','closed_final_split':'test','weights':str(WEIGHTS_DIR/f'{RUN_NAME}_best.pt'),'dataset_yaml':str(yaml_path)}
record_path=DETECTOR_ROOT/'detector_record.json'
record_path.write_text(json.dumps(record,indent=2))
print('='*62)
print('THERMAL DRONE DETECTOR — DEVELOPMENT PIPELINE COMPLETE')
print('='*62)
print('Frozen weights:',record['weights'])
print('Validation summary:',DETECTOR_ROOT/'validation_summary.json')
print('Test split has not been used for model selection.')
print('='*62)
